# vidgen — free GPU server

Runs the text-to-video model on this notebook's free GPU and exposes it through a
free Cloudflare tunnel, so you can drive it from `python -m vidgen ui` on your own computer.

**Pick a GPU first**
- **Google Colab:** *Runtime → Change runtime type → T4 GPU*.
- **Kaggle:** *Settings → Accelerator → GPU T4 x2* and *Settings → Internet → On*
  (needs a phone-verified account). Kaggle gives ~30 GPU hours/week and more RAM — needed for LTX-Video.

Then *Run all*. The last cell prints a **URL** and **TOKEN** — paste them into the local UI
(or into `.env` in your local clone). Keep that cell running; stopping it stops the server.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code
The repo is public, so no token is needed. If you fork it as private, put a GitHub token with read access in `GITHUB_TOKEN`
(Colab: the 🔑 Secrets panel; Kaggle: *Add-ons → Secrets*) — or paste it below.

In [ ]:
import os
REPO = "github.com/oscarneupane/video-gen_free.git"
BRANCH = "main"
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")   # or paste a token here for a private repo
try:  # Colab secrets
    from google.colab import userdata
    GITHUB_TOKEN = GITHUB_TOKEN or userdata.get("GITHUB_TOKEN") or ""
except Exception:
    pass
try:  # Kaggle secrets
    from kaggle_secrets import UserSecretsClient
    GITHUB_TOKEN = GITHUB_TOKEN or UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass

auth = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""
if not os.path.isdir("video-gen_free"):
    !git clone -q --depth 1 -b {BRANCH} https://{auth}{REPO} video-gen_free
%cd video-gen_free

## 2. Install dependencies (~1–2 min)

In [ ]:
!pip install -q -r vidgen/server/requirements.txt

## 3. Start the server
`MODEL` is loaded up front (first run downloads the weights, a few GB):
- `wan-1.3b` — Wan 2.1 1.3B, best quality on a free T4 *(default)*
- `ltx` — LTX-Video, fast 24 fps clips; needs Kaggle's RAM
- `modelscope` — tiny 256px model, handy for a quick end-to-end test

You can still pick any model per video from the UI; switching reloads the weights.

In [ ]:
MODEL = "wan-1.3b"
!python -m vidgen.server.launch --model {MODEL}